In [ ]:
# Core libraries
import os
import glob
import re

# Data handling
import pandas as pd
import numpy as np

# OCR
import easyocr

# cv2
import cv2

In [ ]:
image_folder = r"Image_location_private"

if not os.path.exists(image_folder):
    raise FileNotFoundError(f"Image folder not found: {image_folder}")

In [ ]:
'''image_files = sorted(glob.glob(os.path.join(image_folder, "*.jpg")))

print(f"Number of JPG files found: {len(image_files)}")

for file in image_files:
    print(os.path.basename(file))'''

image_files = sorted(glob.glob(os.path.join(image_folder, "*.jpg")))
# exclude preprocessed files/folder just in case
image_files = [
   f for f in image_files
   if "_preprocessed" not in f and not os.path.basename(f).endswith("_bw.jpg")
]
print(f"Original JPG files found: {len(image_files)}")
for f in image_files:
   print(os.path.basename(f))

In [ ]:
preprocessed_folder = os.path.join(image_folder, "_preprocessed")
os.makedirs(preprocessed_folder, exist_ok=True)
def preprocess_mbs_image(img_path):
   img = cv2.imread(img_path)
   # upscale 2x
   img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
   # grayscale
   gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
   # increase contrast / binary threshold
   _, bw = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY)
   out_path = os.path.join(
       preprocessed_folder,
       os.path.basename(img_path).replace(".jpg", "_bw.jpg")
   )
   cv2.imwrite(out_path, bw)
   return out_path

In [ ]:
try:
    reader = easyocr.Reader(["en"], gpu=True)
    print("EasyOCR reader loaded with GPU enabled.")
except Exception as e:
    print(f"GPU failed, falling back to CPU. Error: {e}")
    reader = easyocr.Reader(["en"], gpu=False)
    print("EasyOCR reader loaded with CPU.")

In [ ]:
full_price_pattern = re.compile(r"^\d{2,3}-\d{2}\+?\s*/\s*\d{1,2}\+?$")
left_price_pattern = re.compile(r"^\d{2,3}-\d{2}\+?$")
right_tick_pattern = re.compile(r"^/?\d{1,2}\+?$")

month_names = "Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec"

row_label_pattern = re.compile(
    rf"^(({month_names})|({month_names})/({month_names}))$",
    re.IGNORECASE
)

header_pattern = re.compile(
    r"^(\d\.\d|UMBS|GNMAII)$",
    re.IGNORECASE
)

In [ ]:
records = []

for img_path in image_files:
    print(f"\n===== Processing Image: {img_path} =====")

    ocr_path = preprocess_mbs_image(img_path)
    print("OCR Input: ", ocr_path)

    result = reader.readtext(
        ocr_path,
        allowlist="0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz-/+.",
        paragraph=False
        )

    for bbox, text, conf in result:
        raw_text = str(text).strip()

        if raw_text == "":
            continue

        (x1, y1), (x2, y2), (x3, y3), (x4, y4) = bbox
        xs = [x1, x2, x3, x4]
        ys = [y1, y2, y3, y4]

        records.append({
            "image": img_path,
            "image_name": os.path.basename(img_path),
            "raw_text": raw_text,
            "conf": conf,
            "x_center": np.mean(xs),
            "y_center": np.mean(ys),
            "x_min": min(xs),
            "x_max": max(xs),
            "y_min": min(ys),
            "y_max": max(ys),
            "bbox": bbox
        })

df_raw = pd.DataFrame(records)

print(f"Raw OCR boxes captured: {len(df_raw)}")
display(df_raw.head(50))

In [ ]:
def normalize_ocr_text(text):
    if not isinstance(text, str):
        return None

    s = text.strip()
    s = s.replace(",", ".")
    s = s.replace("T", "/").replace("t", "/")
    s = s.replace("\\", "/")
    s = re.sub(r"\s+", "", s)

    # Defensive OCR correction:
    # EasyOCR may read slash ticks as 7/71, e.g. /20 -> 720 or 7120.
    # Only convert when the remaining value is a valid MBS tick from 00 to 32.
    slash_misread = re.fullmatch(r"7(?:1)?(\d{2})(\+?)", s)
    if slash_misread:
        tick = slash_misread.group(1)
        plus = slash_misread.group(2)
        if 0 <= int(tick) <= 32:
            s = f"/{tick}{plus}"

    # Same slash correction when attached to a price token:
    # 101-147120 -> 101-14/20
    attached_slash_misread = re.fullmatch(
        r"(\d{2,3}-\d{2}\+?)7(?:1)?(\d{2})(\+?)",
        s
    )
    if attached_slash_misread:
        left_price = attached_slash_misread.group(1)
        tick = attached_slash_misread.group(2)
        plus = attached_slash_misread.group(3)
        if 0 <= int(tick) <= 32:
            s = f"{left_price}/{tick}{plus}"

    # Existing protection: 716 -> /16, 107+ -> /07+
    if re.fullmatch(r"[17]\d{2}\+?", s):
        plus = "+" if s.endswith("+") else ""
        digits = s.replace("+", "")
        tick = digits[1:]
        if tick.isdigit() and 0 <= int(tick) <= 32:
            s = f"/{tick}{plus}"

    return s if s else None


def classify_ocr_text(text):
    if not isinstance(text, str):
        return "other"
    s = text.strip()
    if full_price_pattern.match(s):
        return "full_price"
    if left_price_pattern.match(s):
        return "left_price"
    if right_tick_pattern.match(s):
        return "right_tick"
    if row_label_pattern.match(s):
        return "row_label"
    if header_pattern.match(s):
        return "header"
    return "other"


def split_full_price(text):
    if not isinstance(text, str):
        return None, None

    s = text.strip()

    if "/" not in s:
        return s, None

    price, tick = s.split("/", 1)
    return price, "/" + tick
    
df_ocr = df_raw.copy()
df_ocr["text"] = df_ocr["raw_text"].apply(normalize_ocr_text)
df_ocr["text_type"] = df_ocr["text"].apply(classify_ocr_text)

def assign_row_ids(group, row_tol=30):
   group = group.sort_values("y_center").copy()
   rows = []
   row_centers = []
   for _, r in group.iterrows():
       y = r["y_center"]
       if len(row_centers) == 0:
           row_centers.append(y)
           rows.append(1)
           continue
       distances = [abs(y - yc) for yc in row_centers]
       min_idx = int(np.argmin(distances))
       if distances[min_idx] <= row_tol:
           rows.append(min_idx + 1)
           row_centers[min_idx] = np.mean([row_centers[min_idx], y])
       else:
           row_centers.append(y)
           rows.append(len(row_centers))
   group["row"] = rows
   return group

df_ocr = (
   df_ocr
   .groupby("image", group_keys=False)
   .apply(assign_row_ids)
)
df_ocr = df_ocr.sort_values(["image", "row", "x_center"]).reset_index(drop=True)
display(df_ocr[["image_name", "row", "raw_text", "text", "text_type", "x_center", "y_center"]])


In [ ]:
quote_rows = []
for img_path, img_df in df_ocr.groupby("image"):
   img_df = img_df.sort_values(["row", "x_center"]).copy()
   for row_id, row_df in img_df.groupby("row"):
       row_df = row_df.sort_values("x_center").copy()
       items = row_df[
           row_df["text_type"].isin(["row_label", "full_price", "left_price", "right_tick"])
       ].copy()
       month = None
       pending_price = None
       for _, item in items.iterrows():
           text = item["text"]
           text_type = item["text_type"]
           if text_type == "row_label":
               month = text
               continue
           if text_type == "full_price":
               price, tick = split_full_price(text)
               quote_rows.append({
                   "image_name": os.path.basename(img_path),
                   "row": row_id,
                   "month": month,
                   "price": price,
                   "ticks": tick,
                   "x_center": item["x_center"],
                   "y_center": item["y_center"]
               })
               continue
           if text_type == "left_price":
               pending_price = text
               continue
           if text_type == "right_tick" and pending_price is not None:
               quote_rows.append({
                   "image_name": os.path.basename(img_path),
                   "row": row_id,
                   "month": month,
                   "price": pending_price,
                   "ticks": text,
                   "x_center": item["x_center"],
                   "y_center": item["y_center"]
               })
               pending_price = None

df_final = pd.DataFrame(quote_rows)
df_final = df_final.sort_values(
   ["image_name", "row", "x_center"]
).reset_index(drop=True)
display(df_final)

In [ ]:
def parse_tick_value(tick):
   if not isinstance(tick, str):
       return np.nan
   s = tick.strip().replace("/", "")
   has_plus = s.endswith("+")
   s = s.replace("+", "")
   if not s.isdigit():
       return np.nan
   tick_num = int(s)
   if tick_num < 0 or tick_num > 32:
       return np.nan
   return tick_num + 0.5 if has_plus else tick_num


def parse_price_base(price):
   if not isinstance(price, str):
       return pd.Series([np.nan, np.nan])
   s = price.strip()
   m = re.fullmatch(r"(\d{2,3})-(\d{2})(\+?)", s)
   if m:
       base = int(m.group(1))
       price_tick = int(m.group(2))
       if m.group(3) == "+":
           price_tick += 0.5
       return pd.Series([base, price_tick])
   m = re.fullmatch(r"-(\d{2,3})(\+?)", s)
   if m:
       price_tick = -int(m.group(1))
       if m.group(2) == "+":
           price_tick -= 0.5
       return pd.Series([0, price_tick])
   m = re.fullmatch(r"(\d{1,3})(\+?)", s)
   if m:
       price_tick = int(m.group(1))
       if m.group(2) == "+":
           price_tick += 0.5
       return pd.Series([0, price_tick])
   return pd.Series([np.nan, np.nan])


def calculate_ask_decimal(base, price_tick, ask_tick):
   """
   Convert MBS price/tick fields into decimal price.

   Example:
       price = 101-14, ticks = /20 -> 101 + 20/32 = 101.625

   Important fix:
       The return statement must be outside the rollover IF block.
       Otherwise rows where ask_tick >= price_tick return None/NaN.
   """
   if pd.isna(base) or pd.isna(price_tick) or pd.isna(ask_tick):
       return np.nan

   ask_base = base

   # Rollover case, e.g. 99-30 with /03+ means 100 + 3.5/32.
   if ask_tick < price_tick:
       ask_base += 1

   return ask_base + (ask_tick / 32)


def build_rate_sequence(image_name):
   name = str(image_name)
   if "3.0-5.5" in name:
       return [
           3.0, 3.0,
           4.0, 4.0,
           5.0, 5.0,
           3.5, 3.5,
           4.5, 4.5,
           5.5, 5.5
       ]
   if "4.5-7.0" in name:
       return [
           4.5, 4.5,
           5.5, 5.5,
           6.5, 6.5,
           5.0, 5.0,
           6.0, 6.0,
           7.0, 7.0
       ]
   if "3.0-4.5" in name:
       return [
           3.0,
           4.0,
           3.5,
           4.5
       ]
   if "5.0-6.5" in name:
       return [
           5.0,
           6.0,
           5.5,
           6.5
       ]
   return []


df_final[["price_base", "price_tick"]] = df_final["price"].apply(parse_price_base)
df_final["ask_tick_value"] = df_final["ticks"].apply(parse_tick_value)
df_final["decimal_price"] = df_final.apply(
   lambda r: calculate_ask_decimal(
       r["price_base"],
       r["price_tick"],
       r["ask_tick_value"]
   ),
   axis=1
)

# Audit rows where decimal price did not calculate.
decimal_audit = df_final[df_final["decimal_price"].isna()].copy()
if not decimal_audit.empty:
   print("WARNING: decimal_price did not calculate for these rows:")
   display(decimal_audit[["image_name", "row", "month", "price", "ticks", "price_base", "price_tick", "ask_tick_value"]])
else:
   print("Decimal price audit passed: all rows calculated.")

# Section: top block vs bottom block
df_final["section"] = df_final["row"].apply(lambda r: 1 if r <= 5 else 2)
# Position within each visual row, left to right
df_final["quote_position"] = (
   df_final.groupby(["image_name", "section", "row"]).cumcount() + 1
)
def map_rate(image_name, section, quote_position):
   name = str(image_name)
   if "3.0-5.5" in name:
       rate_seq = [3.5, 3.5, 4.5, 4.5, 5.5, 5.5] if section == 1 else [4.0, 4.0, 4.5, 4.5, 5.5, 5.5]
   elif "4.5-7.0" in name:
       rate_seq = [4.5, 4.5, 5.5, 5.5, 6.5, 6.5] if section == 1 else [5.0, 5.0, 6.0, 6.0, 7.0, 7.0]
   elif "3.0-4.5" in name:
       rate_seq = [3.0, 4.0] if section == 1 else [3.5, 4.5]
   elif "5.0-6.5" in name:
       rate_seq = [5.0, 6.0] if section == 1 else [5.5, 6.5]
   else:
       return np.nan
   i = int(quote_position) - 1
   return rate_seq[i] if i < len(rate_seq) else np.nan

df_final["rate"] = df_final.apply(
   lambda r: map_rate(
       r["image_name"],
       r["section"],
       r["quote_position"]
   ),
   axis=1
)
display(df_final)


In [ ]:
with pd.ExcelWriter('save_location_private', mode = 'a', if_sheet_exists= "replace", engine = 'openpyxl') as writer:
    df_final.to_excel(writer, sheet_name = 'Python New', index = False)